In [3]:
from z3 import *

dim=[4,4]
game="a2c2_2_3b2a2b3" 

dim=[7,7] #34s
game="a5f5_4_2_2b2d3e2a6_2c2_2a2a2a2d3a3b"

dim=[9,9] #3m13s
game="a6e2c4a2a2b2c8_3a4g2_2b5d2a3a3a2a2d4e2a2_2_2_2c2g9_2a"

field=[[]]
game=sum([[int(x)] if ord(x)<=ord('9') else [0]*int(ord(x)-ord('a')+1) for x in game],[])
for s in game:
    if len(field[-1])==dim[0]:
        field.append([])
    field[-1].append(s)
field[-1]+=[0]*(dim[0]-len(field[-1]))
for x in field:
   print(x)

s = Solver()

tilling = Function("tilling", IntSort(),IntSort(),IntSort())


rectangles = []
for x in range(dim[0]):
    for y in range(dim[1]):
        if field[x][y]!=0:
            id = len(rectangles)
            a,b,c,d = Ints("a_%i b_%i c_%i d_%i" % (id,id,id,id))
            rectangles.append((a,b,c,d))

            s.add([a<b,c<d,0<=a,a<=dim[0],0<=b,b<=dim[0],0<=c,c<=dim[1],0<=d,d<=dim[1]])
            #returns unknown without this condition,but can quickly check satisfiability of provided solution and prove that it is only solution
            #normally I excpect that smt solver should be able to find boundaries from other constraints

            s.add((b-a)*(d-c)==field[x][y])
            s.add(tilling(x,y)==id)

            tx,ty = Ints("tx ty")
            s.add(ForAll([tx,ty],Implies(tilling(tx,ty)==id, #And(0<=tx,tx<dim[0],0<=ty,ty<dim[1],tilling(tx,ty)==id),
                                       And(a<=tx,tx<b,c<=ty,ty<d))))

x,y=Ints("x y")
s.add(ForAll([x, y], If(And(0<=x,x<dim[0],0<=y,y<dim[1]), 
                        And(0<=tilling(x,y),tilling(x,y)<len(rectangles)), 
                     tilling(x,y)==-1)))

solution=[
    [0,0,2,3],
    [1,1,2,3],
    [5,4,4,3],
    [5,6,6,6]
]
solution=[]
for x in range(len(solution)):
    for y in range(len(solution[x])):
        s.add(tilling(x,y)==solution[x][y])

rectangles_sol=[
    [0,1,0,2],
    [1,2,0,2],
    [0,2,2,3],
    [0,3,3,4],
    [2,3,1,3],
    [2,4,0,1],
    [3,4,1,4]
]
rectangles_sol=[]
for r in range(len(rectangles_sol)):
    rect = rectangles[r]
    rect_sol = rectangles_sol[r]
    for x in range(len(rect)):
        s.add(rect[x]==rect_sol[x])


while True:
    st=s.check()
    if st==sat:
        m = s.model()

        for x in range(dim[0]):
            for y in range(dim[1]):
                print(m.eval(tilling(x,y)),end='\t')
            print()
        print()
        s.add(Not(And([m.eval(tilling(x,y))==tilling(x,y)  for y in range(dim[1]) for x in range(dim[0]) ])))

    else:
        print(st)
        break

[0, 6, 0, 0, 0, 0, 0, 2, 0]
[0, 0, 4, 0, 2, 0, 2, 0, 0]
[2, 0, 0, 0, 8, 3, 0, 4, 0]
[0, 0, 0, 0, 0, 0, 2, 2, 0]
[0, 5, 0, 0, 0, 0, 2, 0, 3]
[0, 3, 0, 2, 0, 2, 0, 0, 0]
[0, 4, 0, 0, 0, 0, 0, 2, 0]
[2, 2, 2, 2, 0, 0, 0, 2, 0]
[0, 0, 0, 0, 0, 0, 9, 2, 0]
0	0	0	0	0	0	4	1	1	
2	2	2	2	3	3	4	8	8	
5	6	6	6	6	7	9	8	8	
5	6	6	6	6	7	9	10	10	
11	11	11	11	11	7	12	12	13	
14	14	14	15	15	16	16	18	13	
17	17	17	17	24	24	24	18	13	
19	20	21	22	24	24	24	23	23	
19	20	21	22	24	24	24	25	25	

unsat
